In [ ]:
import os
import sys

# Drive Setting
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/kyul_stt"

assert os.path.isdir(PROJECT_ROOT), PROJECT_ROOT

SRC = os.path.join(PROJECT_ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(PROJECT_ROOT) # 작업 디렉토리 변경

print("cwd:", os.getcwd()) # 파이썬이 작업 디렉토리로 잡고 있는 절대 경로
print("src in path:", SRC)

Mounted at /content/drive
cwd: /content/drive/MyDrive/Colab Notebooks/kyul_stt
src in path: /content/drive/MyDrive/Colab Notebooks/kyul_stt/src


In [ ]:
import os

# csv & wav path setting
# csv path
CSV_PATH = os.path.join(PROJECT_ROOT, "data", "5_csv", "5차년도_2차.csv")
ENCODING = "cp949"
# wav path
WAV_DIR = os.path.join(PROJECT_ROOT, "data", "5th_2nd") # Changed WAV_DIR path
WAV_PATTERN = "{wav_id}.wav"

assert os.path.isfile(CSV_PATH)
assert os.path.isdir(WAV_DIR)

In [ ]:
OUT_DIR = os.path.join(PROJECT_ROOT, "runs", "ser_aihub_nb") # 학습 결과 디렉토리
BASE_MODEL = "kresnik/wav2vec2-large-xlsr-korean" # 베이스 모델

In [ ]:
# Hyper Parameter Tuning
SECONDS = 3.0 
EPOCHS = 3
BATCH_SIZE = 2
LR = 1e-4
SEED = 42

In [ ]:
from __future__ import annotations

import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from kyul.compat import disable_tensorflow_for_transformers

disable_tensorflow_for_transformers()

from transformers import (
    AutoConfig,
    Trainer,
    TrainingArguments,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
)

# Hugging Face Transforemers가 TensorFlow(TF)를 불러오지 못하게 만든다.
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")

# AI-Hub 7감정 (영문) → 숫자 라벨(0~6)로 변환하기 위한
LABELS_EN_7 = [
    "happiness",
    "angry",
    "disgust",
    "fear",
    "neutral",
    "sadness",
    "surprise",
]

# label_2_id
LABEL2ID = {
    "happiness": 0,
    "angry": 1,
    "disgust": 2,
    "fear": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6,
}

# id_2_label
ID2LABEL = {
    0: "happiness",
    1: "angry",
    2: "disgust",
    3: "fear",
    4: "neutral",
    5: "sadness",
    6: "surprise",
}

# normalize_situation
# -> 공백 제거, 대소문자 통일,
def normalize_situation(raw):

    # NaN(숫자 결측)값 None
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None

    # 어떤 타입이든 문자열로 바꾼다.
    # strip() - 공백 제거
    # lower() - 대소문자 통일
    t = str(raw).strip().lower()

    if not t:
        return None

    syn = {
        "anger": "angry",
        "angry": "angry",
        "happiness": "happiness",
        "happy": "happiness",
        "disgust": "disgust",
        "fear": "fear",
        "neutral": "neutral",
        "sadness": "sadness",
        "sad": "sadness",
        "surprise": "surprise",
    }

    return syn.get(t)


# set_seed - 무작위 시드 고정하는 함수
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_audio_16k_mono(path: str, target_sr: int = 16000) -> np.ndarray:
    data, sr = sf.read(path, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != target_sr:
        n = int(len(data) * target_sr / sr)
        data = resample(data, n).astype(np.float32)
    return np.asarray(data, dtype=np.float32)


def fix_to_seconds(audio: np.ndarray, seconds: float, sr: int = 16000) -> np.ndarray:
    target_len = int(sr * seconds)
    if len(audio) >= target_len:
        return audio[:target_len]
    out = np.zeros(target_len, dtype=np.float32)
    out[: len(audio)] = audio
    return out


def resolve_wav_path(wav_dir: Path, wav_id: str, pattern: str) -> Path:
    return wav_dir / pattern.replace("{wav_id}", str(wav_id).strip())


def load_rows(csv_path: Path, wav_dir: Path, encoding: str, pattern: str) -> list[dict[str, Any]]:
    df = pd.read_csv(csv_path, encoding=encoding)
    rows = []
    sk_l, sk_m = 0, 0
    for _, r in df.iterrows():
        lab = normalize_situation(r.get("상황"))
        if lab is None or lab not in LABEL2ID:
            sk_l += 1
            continue
        wid = str(r["wav_id"]).strip()
        wp = resolve_wav_path(wav_dir, wid, pattern)
        if not wp.is_file():
            sk_m += 1
            continue
        rows.append(
            {"wav_path": str(wp), "label": LABEL2ID[lab], "label_en": lab, "wav_id": wid}
        )
    print(f"유효 {len(rows)} / 전체 {len(df)} (라벨스킵 {sk_l}, 파일없음 {sk_m})")
    return rows


set_seed(SEED)

rows = load_rows(Path(CSV_PATH), Path(WAV_DIR), ENCODING, WAV_PATTERN)
if not rows:
    raise RuntimeError("유효 샘플 0 — WAV_DIR·WAV_PATTERN·CSV 확인")

print("예시:", rows[0])

유효 12333 / 전체 19374 (라벨스킵 0, 파일없음 7041)
예시: {'wav_path': '/content/drive/MyDrive/Colab Notebooks/kyul_stt/data/5th_2nd/5f4141e29dd513131eacee2f.wav', 'label': 0, 'label_en': 'happiness', 'wav_id': '5f4141e29dd513131eacee2f'}


In [ ]:
print(f"Listing contents of WAV_DIR: {WAV_DIR}")
# List first 10 files/directories in WAV_DIR to check its structure
import os
import glob

# Get a list of all files and directories directly under WAV_DIR
items_in_wav_dir = sorted(os.listdir(WAV_DIR))

# Print the first 10 items, or fewer if there aren't 10
print("First 10 items in WAV_DIR:")
for item in items_in_wav_dir[:10]:
    full_path = os.path.join(WAV_DIR, item)
    if os.path.isdir(full_path):
        print(f"  [DIR] {item}")
    elif os.path.isfile(full_path):
        print(f"  [FILE] {item}")
    else:
        print(f"  [OTHER] {item}")

# Optionally, check for .wav files in the root of WAV_DIR
wav_files_in_root = glob.glob(os.path.join(WAV_DIR, "*.wav"))
if wav_files_in_root:
    print(f"\nFound {len(wav_files_in_root)} .wav files directly in {WAV_DIR}. Example: {os.path.basename(wav_files_in_root[0])}")
else:
    print(f"\nNo .wav files found directly in {WAV_DIR}. They might be in subdirectories.")

Listing contents of WAV_DIR: /content/drive/MyDrive/Colab Notebooks/kyul_stt/data/5th_2nd
First 10 items in WAV_DIR:
  [FILE] 5f3c9ed98a3c1005aa97c4bd.wav
  [FILE] 5f3c9ef78a3c1005aa97c4be.wav
  [FILE] 5f3c9f658a3c1005aa97c4c7.wav
  [FILE] 5f3c9f808a3c1005aa97c4c8.wav
  [FILE] 5f3c9f9c8a3c1005aa97c4cb.wav
  [FILE] 5f3c9fcc8a3c1005aa97c4ce.wav
  [FILE] 5f3ca01b8a3c1005aa97c4d3.wav
  [FILE] 5f3ca06b8a3c1005aa97c4da.wav
  [FILE] 5f3ca09d8a3c1005aa97c4df.wav
  [FILE] 5f3ca4778a3c1005aa97c50d.wav

Found 12333 .wav files directly in /content/drive/MyDrive/Colab Notebooks/kyul_stt/data/5th_2nd. Example: 5f97ee4a111dfd48d40ff590.wav


In [ ]:
import pandas as pd

# Load the CSV again to inspect the 'wav_id' column
df_check = pd.read_csv(CSV_PATH, encoding=ENCODING)

print(f"\nSample of 'wav_id' column from {CSV_PATH}:")
display(df_check['wav_id'].head())

print(f"\nNumber of unique 'wav_id' values: {df_check['wav_id'].nunique()}")

# Check for common prefixes/suffixes or inconsistent formats
# For example, checking if 'wav_id' contains '.wav' already
sample_wav_ids = df_check['wav_id'].astype(str).sample(min(5, len(df_check)), random_state=SEED).tolist()
print(f"\nRandom sample of 5 wav_ids: {sample_wav_ids}")

contains_wav_extension = df_check['wav_id'].astype(str).str.contains('.wav', case=False, na=False).any()
print(f"Are there any 'wav_id' entries containing '.wav' extension?: {contains_wav_extension}")

# Check for any leading/trailing spaces in wav_id
contains_extra_spaces = df_check['wav_id'].astype(str).apply(lambda x: x != x.strip()).any()
print(f"Are there any 'wav_id' entries with leading/trailing spaces?: {contains_extra_spaces}")


Sample of 'wav_id' column from /content/drive/MyDrive/Colab Notebooks/kyul_stt/data/5_csv/5차년도_2차.csv:


,wav_id
0,5f4141e29dd513131eacee2f
1,5f4141f59dd513131eacee30
2,5f4142119dd513131eacee31
3,5f4142279dd513131eacee32
4,5f3c9ed98a3c1005aa97c4bd



Number of unique 'wav_id' values: 19374

Random sample of 5 wav_ids: ['5f69f23a111dfd48d40fce7e', '5f6fff92f8fac448cc0a618c', '5f69a3bcd338b948c4e684a4', '5fb384270fb0f33aa31ca287', '5fbc9d534c55eb78bd7ce900']
Are there any 'wav_id' entries containing '.wav' extension?: False
Are there any 'wav_id' entries with leading/trailing spaces?: False


In [ ]:
# train/validation/test split - 이 부분은 SCTniU-E7tll 셀의 split_stratified 함수로 이미 처리되었습니다.
train_val_rows, test_rows = train_test_split(
    rows, test_size=0.1, random_state=SEED, stratify=[r['label'] for r in rows]
)
train_rows, val_rows = train_test_split(
    train_val_rows, test_size=(0.1 / 0.9), random_state=SEED, stratify=[r['label'] for r in train_val_rows]
)

# train_rows, val_rows, test_rows 변수는 SCTniU-E7tll 셀에서 이미 설정되었습니다.
print(f"Train set size: {len(train_rows)}")
print(f"Validation set size: {len(val_rows)}")
print(f"Test set size: {len(test_rows)}")

Train set size: 9865
Validation set size: 1234
Test set size: 1234


In [ ]:
class SERDataset(torch.utils.data.Dataset):
    def __init__(self, rows, feature_extractor, seconds: float):
        self.rows = rows
        self.fe = feature_extractor
        self.seconds = seconds

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int):
        r = self.rows[idx]
        audio = load_audio_16k_mono(r["wav_path"])
        audio = fix_to_seconds(audio, SECONDS)
        x = self.fe(audio, sampling_rate=16000, return_tensors="pt", padding=False)
        return {
            "input_values": x["input_values"].squeeze(0),
            "labels": torch.tensor(int(r["label"]), dtype=torch.long),
        }


@dataclass
class Collator:
    feature_extractor: Any

    def __call__(self, features):
        input_values = [f["input_values"] for f in features]
        labels = torch.stack([f["labels"] for f in features])
        batch = self.feature_extractor.pad(
            {"input_values": input_values}, padding=True, return_tensors="pt"
        )
        batch["labels"] = labels
        return batch


# Lg
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }


def split_stratified(rows, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    if len(rows) < 3:
        return rows, [], []
    y = [int(r["label"]) for r in rows]
    try:
        train_rows, tmp_rows = train_test_split(
            rows, test_size=(1.0 - train_ratio), random_state=seed, stratify=y
        )
        vf = val_ratio / (val_ratio + test_ratio) if (val_ratio + test_ratio) > 0 else 0.5
        y_tmp = [int(r["label"]) for r in tmp_rows]
        val_rows, test_rows = train_test_split(
            tmp_rows, test_size=(1.0 - vf), random_state=seed, stratify=y_tmp
        )
    except ValueError:
        rng = random.Random(seed)
        tmp = rows[:]
        rng.shuffle(tmp)
        n = len(tmp)
        n_train = max(1, int(n * train_ratio))
        n_val = max(1, int(n * val_ratio))
        train_rows, val_rows, test_rows = tmp[:n_train], tmp[n_train : n_train + n_val], tmp[n_train + n_val :]
        if not test_rows:
            test_rows = val_rows
    return train_rows, val_rows, test_rows


train_rows, val_rows, test_rows = split_stratified(rows, seed=SEED)
print("train", len(train_rows), "val", len(val_rows), "test", len(test_rows))

train 8633 val 1850 test 1850


In [ ]:
fe = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)
cfg = AutoConfig.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS_EN_7),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    finetuning_task="ser_aihub_nb",
)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL, config=cfg, ignore_mismatched_sizes=True
)

train_ds = SERDataset(train_rows, fe, SECONDS)
val_ds = SERDataset(val_rows, fe, SECONDS)
test_ds = SERDataset(test_rows, fe, SECONDS)
collator = Collator(fe)

out_sub = Path(OUT_DIR) / f"{Path(BASE_MODEL).name}_sec{SECONDS:g}"
out_sub.mkdir(parents=True, exist_ok=True)

targs = TrainingArguments(
    output_dir=str(out_sub),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    num_train_epochs=EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=targs,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train() # 학습 시작

def epoch_metrics_table(trainer):
    import pandas as pd

    logs = trainer.state.log_history
    eval_logs = [x for x in logs if "eval_loss" in x]
    train_logs = [x for x in logs if "loss" in x and "eval_loss" not in x]
    rows = []
    prev_e = 0.0
    for ev in sorted(eval_logs, key=lambda x: float(x["epoch"])):
        e = float(ev["epoch"])
        chunk = [t["loss"] for t in train_logs if prev_e <= float(t.get("epoch", 0)) < e]
        train_loss = sum(chunk) / len(chunk) if chunk else float("nan")
        rows.append(
            {
                "Epoch": int(round(e)),
                "Training Loss": train_loss,
                "Validation Loss": ev["eval_loss"],
                "Accuracy": ev.get("eval_accuracy", float("nan")),
                "Macro F1": ev.get("eval_macro_f1", float("nan")),
            }
        )
        prev_e = e
    return pd.DataFrame(rows)


summary_df = epoch_metrics_table(trainer)
try:
    from IPython.display import display

    display(summary_df)
except Exception:
    print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

print("val:", trainer.evaluate(eval_dataset=val_ds))
print("test:", trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test"))

MODEL_DIR = out_sub / "model"
trainer.save_model(str(MODEL_DIR))
fe.save_pretrained(str(MODEL_DIR))
print("저장 완료:", MODEL_DIR)
print("④에서 사용:", str(MODEL_DIR))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: kresnik/wav2vec2-large-xlsr-korean
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
projector.weight  | MISSING    | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 
projector.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
